# Data Loading

In [27]:
%%time

import kplex_dev
import data_splits
import warnings
import pandas as pd
import numpy as np
import ast
from sklearn.preprocessing import LabelEncoder


warnings.filterwarnings('ignore')

timeout_replacement = 1800  # max timeout replacement
excel_path = "kplex_ML_data.xlsx"


CPU times: total: 0 ns
Wall time: 0 ns


In [28]:
# Loading and preprocessing data
cat_df, dfs = kplex_dev.load_excel_data(excel_path, timeout_replacement)
kplex_dev.process_kcore_list_to_statistics(cat_df, keep_col=True)

# Selecting weight scheme
epsilon = 0.001 # need to set !!!
scheme = kplex_dev.W_TIME_MIN_X_W_LOG_TIME_RATIO # Change this and analyse!!

# try dense cross entrpy loss for NN
# take 10% of data out as unseen data and run triaing and test split on remaining 90%data

# Assigning weights
kplex_dev.assign_weights(cat_df, scheme, epsilon)

# kplex_dev.export_sorted(cat_df)

# Data reformatting
cat_df = kplex_dev.post_process_data(cat_df)
cat_df['kcore_list'] = cat_df['kcore_list'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# fastest_time = minimum across those columns
cat_df['fastest_time'] = cat_df[data_splits.algorithms].min(axis=1)

cat_df

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,...,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br,kcore_list,fastest_time
0,MANN-a81,20,3321,5506380,1658.0488,3317,3280,3280,1,3321.000000,...,3321,0.000000,306.046574,1459.5,1800.0,1468.1,1477.4,1504.7,"[(3280, 3321)]",1459.5
1,wikipedia_link_sr,20,1810,1637059,904.4525,1809,1798,1795,4,1806.250000,...,1803,2.861381,66.902739,22.6,18.0,737.7,740.4,738.1,"[(1795, 1810), (1796, 1808), (1797, 1804), (17...",18.0
2,scc_reality,20,1332,880733,661.2110,1331,1235,1230,6,1290.333333,...,1236,35.714921,47.684856,1800.0,1214.8,9.4,9.7,9.0,"[(1230, 1332), (1231, 1330), (1232, 1309), (12...",9.0
3,web-wikipedia_link_en13-all,20,432,93085,215.4745,431,427,427,1,432.000000,...,432,0.000000,184.662528,239.3,287.1,517.7,504.4,516.7,"[(427, 432)]",239.3
4,livejournal-links,20,442,94435,213.6538,441,371,369,3,412.333333,...,386,22.983086,0.269931,1.3,1.6,1.6,1.6,1.6,"[(369, 442), (370, 409), (371, 386)]",1.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,2,264,11792,44.6667,150,67,52,16,215.000000,...,125,56.656862,0.784308,2.0,0.6,0.5,0.5,2.4,"[(52, 264), (53, 262), (54, 261), (55, 260), (...",0.5
756,edit-ukwiki,2,281,14183,50.4733,280,65,40,26,235.384615,...,168,34.703222,0.437734,1.2,1.1,0.6,0.6,0.5,"[(40, 281), (41, 279), (42, 275), (43, 271), (...",0.5
757,johnson8-4-4,2,70,1855,26.5000,53,53,53,1,70.000000,...,70,0.000000,0.389785,0.8,0.9,1.1,1.1,0.3,"[(53, 70)]",0.3
758,hamming6-2,2,64,1824,28.5000,57,57,57,1,64.000000,...,64,0.000000,0.081093,0.2,0.3,0.2,0.2,0.3,"[(57, 64)]",0.2


# Training with LSTM

Simple LSTM network with a soft‑cross‑entropy loss.


### Model

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.metrics import classification_report

In [30]:
# --- 1) Soft Cross‐Entropy with Temperature --------------------------

class SoftCrossEntropyLoss(nn.Module):
    """
    Cross‐entropy loss where logits are divided by a temperature T before softmax.
    """
    def __init__(self, temperature: float = 1.0):
        super().__init__()
        self.temperature = temperature

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        # scale logits
        scaled = logits / self.temperature
        return F.cross_entropy(scaled, targets)

In [31]:
# kplex_dev.py

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted

# ——— 1) Soft Cross‐Entropy with Temperature ——————————————

class SoftCrossEntropyLoss(nn.Module):
    def __init__(self, temperature: float = 1.0):
        super().__init__()
        self.temperature = temperature
    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        scaled = logits / self.temperature
        return F.cross_entropy(scaled, targets)

# ——— 2) Custom Dataset to yield features + k_core sequences ——————
import ast
import numpy as np
import torch
from torch.utils.data import Dataset

class KCoreDataset(Dataset):
    def __init__(self, X, y, k_core_list, scaler):
        assert len(X) == len(y) == len(k_core_list), (
            f"X ({len(X)}), y ({len(y)}), and k_core_list ({len(k_core_list)}) must all have the same length"
        )
        # 1) Scale features once
        self.X = scaler.transform(X)
        self.y = np.array(y, dtype=int)
        
        # 2) Parse & cache sequences as numeric arrays
        self.seqs = []
        for entry in k_core_list:
            parsed = entry
            # If it's a string (including numpy.str_), un-wrap until we get a list
            while isinstance(parsed, str):
                parsed = ast.literal_eval(parsed)
            # Now parsed should be a list of (int, int) tuples
            arr = np.array(parsed, dtype=float)  # shape (L,2)
            self.seqs.append(arr)
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        features = torch.tensor(self.X[idx], dtype=torch.float32)
        target   = torch.tensor(self.y[idx], dtype=torch.long)
        seq_arr  = self.seqs[idx]                         # np.ndarray (L,2)
        seq      = torch.tensor(seq_arr, dtype=torch.float32)
        return features, seq, target


from torch.nn.utils.rnn import pad_sequence

# get a nice tensor with equal size, straightforward indexing, and LSTM just reads zeros on the padded steps.
def pad_collate(batch):
    """
    batch is a list of (features, seq, target) tuples.
    We pad the seqs along time‐axis so they all match the longest in this batch.
    """
    feats, seqs, tgts = zip(*batch)

    # Stack features and targets normally
    feats = torch.stack(feats, 0)
    tgts  = torch.stack(tgts,  0)

    # Pad the sequences (each is [L_i, 2]) to shape [L_max, 2]
    # pad_sequence wants a list of [L_i,2] tensors
    padded = pad_sequence(seqs, batch_first=True, padding_value=0.0)
    # padded.shape == (batch_size, L_max, 2)

    return feats, padded, tgts



# ——— 3) LSTM‐based classifier ——————————————————————————

class TorchLSTMClassifier(BaseEstimator, ClassifierMixin):
    """
    sklearn‐compatible PyTorch LSTM classifier that
    incrementally leaks k_core_list tuples over epochs.
    """
    def __init__(
        self,
        hidden_layers=(128,64),
        lstm_hidden=32,
        lr: float = 1e-2,
        batch_size: int = 64,
        epochs: int = 50,
        temperature: float = 1.0,
        device: str = None,
        random_state: int = 42
    ):
        self.hidden_layers = hidden_layers
        self.lstm_hidden   = lstm_hidden
        self.lr            = lr
        self.batch_size    = batch_size
        self.epochs        = epochs
        self.temperature   = temperature
        self.device        = device
        self.random_state  = random_state

    def _build_model(self, n_feat, n_classes):
        # feature MLP
        mlp_layers = []
        in_dim = n_feat
        for h in self.hidden_layers:
            mlp_layers += [nn.Linear(in_dim, h), nn.ReLU()]
            in_dim = h
        self.mlp = nn.Sequential(*mlp_layers)
        # LSTM encoder for k_core tuples
        self.lstm = nn.LSTM(input_size=2, hidden_size=self.lstm_hidden, batch_first=True)
        # final classifier
        tot_dim = in_dim + self.lstm_hidden
        self.classifier = nn.Linear(tot_dim, n_classes)
        return

    def fit(self, X, y):
        # X: DataFrame of features
        # y: 1d array
        
        k_core_list = [np.array(tuples_list) for tuples_list in X['kcore_list']]
        X = X.drop('kcore_list', axis = 1)

        X_arr, y_arr = check_X_y(X.values, y)
        self.classes_ = np.unique(y_arr)
        n_classes = len(self.classes_)
        n_feat = X_arr.shape[1]

        torch.manual_seed(self.random_state)
        self.device_ = torch.device(self.device or ('cuda' if torch.cuda.is_available() else 'cpu'))

        # scale features
        self.scaler_ = StandardScaler().fit(X_arr)

        # build model
        self._build_model(n_feat, n_classes)
        self.mlp.to(self.device_)
        self.lstm.to(self.device_)
        self.classifier.to(self.device_)

        criterion = SoftCrossEntropyLoss(self.temperature)
        optimizer = optim.SGD(
            list(self.mlp.parameters()) +
            list(self.lstm.parameters()) +
            list(self.classifier.parameters()),
            lr=self.lr
        )

        # prepare dataset

        ds = KCoreDataset(X_arr, y_arr, k_core_list, self.scaler_)
        # print('dataset', ds)
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=True, collate_fn=pad_collate)

        for epoch in range(1, self.epochs+1):
            self.mlp.train(); self.lstm.train(); self.classifier.train()
            for feats, seq, targets in loader:
                feats   = feats.to(self.device_)
                seq     = seq.to(self.device_)             # shape B x L x 2 ; data points x length of kcore list x tuple size
                targets = targets.to(self.device_)
                optimizer.zero_grad()

                # leak only first `epoch` tuples
                max_len = min(epoch, seq.size(1))
                truncated = seq[:, :max_len, :]            # B x max_len x 2
                # pack/pad if needed
                out, (h_n, _) = self.lstm(truncated)
                lstm_feat = h_n[-1]                         # B x lstm_hidden

                mlp_feat = self.mlp(feats)                  # B x last_hidden

                combined = torch.cat([mlp_feat, lstm_feat], dim=1)
                logits = self.classifier(combined)          # B x n_classes

                loss = criterion(logits, targets)
                loss.backward()
                optimizer.step()
            # optional: print epoch
            # print(f"Epoch {epoch}/{self.epochs}")
        return self

    def predict(self, X):
        check_is_fitted(self, ['mlp','lstm','classifier','scaler_'])
        
        k_core_list = [np.array(tuples_list) for tuples_list in X['kcore_list']]
        X = X.drop('kcore_list', axis = 1)
        
        X_arr = check_array(X.values)
        # print('HEREHHHH')

        # ds = KCoreDataset(X_arr, np.zeros(len(X_arr)), self.k_core_list, self.scaler_)
        ds = KCoreDataset(X_arr, np.zeros(len(X_arr)), k_core_list, self.scaler_)
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=False, collate_fn=pad_collate)

        preds = []
        self.mlp.eval(); self.lstm.eval(); self.classifier.eval()
        with torch.no_grad():
            for feats, seq, _ in loader:
                feats = feats.to(self.device_)
                seq   = seq.to(self.device_)
                # leak full sequence at inference
                out, (h_n, _) = self.lstm(seq)
                lstm_feat = h_n[-1]
                mlp_feat  = self.mlp(feats)
                combined  = torch.cat([mlp_feat, lstm_feat], dim=1)
                logits    = self.classifier(combined)
                p         = logits.argmax(dim=1).cpu().numpy()
                preds.append(p)
        preds = np.concatenate(preds)
        inv_map = {i:c for i,c in enumerate(self.classes_)}
        return np.vectorize(inv_map.get)(preds)

    def predict_proba(self, X):
        # similar to predict(), but softmax the logits / temperature
        check_is_fitted(self, ['mlp','lstm','classifier','scaler_'])
        
        k_core_list = [np.array(tuples_list) for tuples_list in X['kcore_list']]
        X = X.drop('kcore_list', axis = 1)

        X_arr = check_array(X.values)
        ds = KCoreDataset(X_arr, np.zeros(len(X_arr)), k_core_list, self.scaler_)
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=False, collate_fn=pad_collate)

        probs_list = []
        self.mlp.eval(); self.lstm.eval(); self.classifier.eval()
        with torch.no_grad():
            for feats, seq, _ in loader:
                feats = feats.to(self.device_)
                seq   = seq.to(self.device_)
                out, (h_n, _) = self.lstm(seq)
                lstm_feat = h_n[-1]
                mlp_feat  = self.mlp(feats)
                combined  = torch.cat([mlp_feat, lstm_feat], dim=1)
                logits    = self.classifier(combined) / self.temperature
                probs     = F.softmax(logits, dim=1).cpu().numpy()
                probs_list.append(probs)
        return np.vstack(probs_list)


In [32]:
def sanity_check_dataset(ds, n_samples=5):
    N = len(ds)
    print(f"Dataset size: {N} samples")

    # 1) feature dimensionality
    feats0, seq0, tgt0 = ds[0]
    print(f"Feature vector dim: {feats0.numel()}, Seq[0] shape: {tuple(seq0.shape)}, Example label: {int(tgt0)}")

    # 2) label distribution
    all_targets = [int(ds[i][2]) for i in range(N)]
    print("\nLabel distribution:")
    print(pd.Series(all_targets).value_counts(normalize=True))

    # 3) sequence length stats
    lengths = [ds[i][1].shape[0] for i in range(N)]
    print("\nSequence length (L) stats:", 
          f"min={np.min(lengths)}, max={np.max(lengths)}, mean={np.mean(lengths):.2f}")

    # 4) peek at first few samples
    print(f"\nFirst {min(n_samples,N)} samples:")
    for i in range(min(n_samples, N)):
        feats, seq, tgt = ds[i]
        print(f" [{i:>2}] feat[:3]={feats[:3].numpy()}, seq.shape={tuple(seq.shape)}, tgt={int(tgt)}")


## Tie Breaking and Data Splitting

### **Balanced train-test split based on Y**

Normal training using train-test-split with 20% of data as test data

In [33]:
from data_splits import prepare_train_test_strat_Y
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y(cat_df, test_size=0.2, 
    kcore_list=True, tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
    )

# Label‐encode the string targets into integers 0…4  ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

X_train_ = X_train.copy()


Expanded 687 rows with near ties using tolerance 50.0%.
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


##### **Sanity‑check for any KCoreDataset instance**

In [34]:
# after you’ve constructed your dataset:
k_core_list_train = [np.array(tuples_list) for tuples_list in X_train['kcore_list']]
X_train_copy = X_train.drop('kcore_list', axis = 1)

scaler = StandardScaler().fit(X_train_copy.values)
dataset = KCoreDataset(X_train_copy.values, y_train, k_core_list_train, scaler)
sanity_check_dataset(dataset)


Dataset size: 1677 samples
Feature vector dim: 12, Seq[0] shape: (30, 2), Example label: 2

Label distribution:
4    0.277281
1    0.272510
2    0.208110
0    0.123435
3    0.118664
Name: proportion, dtype: float64

Sequence length (L) stats: min=1, max=184, mean=29.64

First 5 samples:
 [ 0] feat[:3]=[ 0.6256904  -0.16007867 -0.2722594 ], seq.shape=(30, 2), tgt=2
 [ 1] feat[:3]=[ 0.6256904  -0.15869237 -0.15993318], seq.shape=(1, 2), tgt=0
 [ 2] feat[:3]=[ 0.4083576  -0.15869237 -0.15993318], seq.shape=(1, 2), tgt=4
 [ 3] feat[:3]=[-0.02630803 -0.15819329 -0.255955  ], seq.shape=(49, 2), tgt=2
 [ 4] feat[:3]=[-0.24364083 -0.16028602 -0.26144662], seq.shape=(9, 2), tgt=4


🧐 Data Sanity Check Summary

Before training, we verify that our dataset and sequence pipeline are correct:

1. **Feature Vectors (12‑dim)**  
   - All 12 input features are **standardized** (zero mean, unit variance) to ensure stable, balanced optimization.

2. **K‑Core Sequences (variable L×2)**  
   - Example `Seq[0] shape: (6, 2)` means the first row has 6 `(k_value, size)` tuples.  
   - Overall lengths range **min = 1**, **max = 184**, **mean ≈ 26**.  
   - Our LSTM will “leak in” these tuples gradually over epochs.

3. **Label Distribution**  
    - About 34% of all samples are class 4, 24% are class 2, and so on. This ensures our split and any weighting strategy can account for the fact that some algorithms appear much more often than others.

3. **Sequence Length Statistics**

min=1, max=184, mean=26.11

Some graphs have only one core tuple, others up to 184. On average there are about 26 tuples per row. Our LSTM will gradually “leak in” more of these per epoch, but it’s good to know the range ahead of time.


🧐 Data Sanity Check Summary

Before training, we verify that our dataset and sequence pipeline are correct:

1. **Feature Vectors (12‑dim)**  
   - All 12 input features are **standardized** (zero mean, unit variance) to ensure stable, balanced optimization.

2. **K‑Core Sequences (variable L×2)**  
   - Example `Seq[0] shape: (6, 2)` means the first row has 6 `(k_value, size)` tuples.  
   - Overall lengths range **min = 1**, **max = 184**, **mean ≈ 26**.  
   - Our LSTM will “leak in” these tuples gradually over epochs.

3. **Label Distribution**  
    - About 34% of all samples are class 4, 24% are class 2, and so on. This ensures our split and any weighting strategy can account for the fact that some algorithms appear much more often than others.

3. **Sequence Length Statistics**

min=1, max=184, mean=26.11

Some graphs have only one core tuple, others up to 184. On average there are about 26 tuples per row. Our LSTM will gradually “leak in” more of these per epoch, but it’s good to know the range ahead of time.


**Training**

In [35]:
# Before calling fit():
clf_lstm_strat_y = TorchLSTMClassifier(hidden_layers=(128, 64, 32), lstm_hidden=32, epochs=1000, temperature=1.0, lr= 0.01)
# supply your sequence-of-(k,start,end) tuples per row
clf_lstm_strat_y.k_core_list = [np.array(tuples_list) for tuples_list in X_train_['kcore_list']]

clf_lstm_strat_y.fit(X_train, y_train)


TorchLSTMClassifier(epochs=1000, hidden_layers=(128, 64, 32))

#### Inference on 20% dataset

In [36]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=clf_lstm_strat_y,
    le=le,
    kcore_list = True,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LSTM_results/tie_tol/50/expand_and_split/',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LSTM_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 47.14%

Precision by Algorithm:
  UMKP: 58.33% (12 predictions)
  AltRB: 35.90% (78 predictions)
  kPEX: 66.67% (156 predictions)
  Pivot_Br: 19.18% (73 predictions)
  S_Br: 44.55% (101 predictions)

Recall by Algorithm:
  UMKP: 46.94% (23/49 cases)
  AltRB: 66.67% (40/60 cases)
  kPEX: 58.60% (126/215 cases)
  Pivot_Br: 57.42% (89/155 cases)
  S_Br: 55.70% (83/149 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 63346.10
  Total Optimal Time   = 48282.00
  Time Ratio (pred/opt)= 1.312
  Avg Overhead/case ((pred - opt) / opt)    = 2721.06%


Adding prediction probabilities...

Saving full predictions to './LSTM_results/tie_tol/50/expand_and_split/all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 222 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
           dataset  k_value 

#### Re-evalute on entire dataset

In [37]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_lstm_strat_y,
    le=le,
    kcore_list = True,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LSTM_results/tie_tol/50/expand_and_split/',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LSTM_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 46.71%

Precision by Algorithm:
  UMKP: 53.57% (28 predictions)
  AltRB: 43.70% (135 predictions)
  kPEX: 61.35% (282 predictions)
  Pivot_Br: 16.46% (158 predictions)
  S_Br: 52.23% (157 predictions)

Recall by Algorithm:
  UMKP: 45.00% (45/100 cases)
  AltRB: 70.69% (82/116 cases)
  kPEX: 57.02% (207/363 cases)
  Pivot_Br: 60.62% (137/226 cases)
  S_Br: 55.08% (141/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 102912.20
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.539
  Avg Overhead/case ((pred - opt) / opt)    = 226.19%


Adding prediction probabilities...

Saving full predictions to './LSTM_results/tie_tol/50/expand_and_split/all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 405 wrong predictions with time metrics


### **Balanced train-test split based on both Y and k**

In [38]:
from data_splits import prepare_train_test_strat_Y_K
from sklearn.preprocessing import LabelEncoder

X_train, X_test, y_train_str, y_test_str, weights_train, weights_test = prepare_train_test_strat_Y_K(cat_df, test_size=0.2, 
        kcore_list=True, tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
    )

# Label‐encode the string targets into integers 0…4  ---
le = LabelEncoder()
y_train = le.fit_transform(y_train_str)     # e.g. ['AltRB','UMKP',…] → [0,3,…]
y_test  = le.transform(y_test_str)          # uses the same mapping

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

X_train_ = X_train.copy()



Expanded 687 rows with near ties using tolerance 50.0%.
Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [39]:
# Instantiate and train
# Before calling fit():
clf_lstm_strat_y_K = TorchLSTMClassifier(hidden_layers=(128, 64, 32), lstm_hidden=32, epochs=500, temperature=1.0, lr= 0.01)
# supply your sequence-of-(k,start,end) tuples per row
clf_lstm_strat_y_K.k_core_list = [np.array(tuples_list) for tuples_list in X_train_['kcore_list']]

clf_lstm_strat_y_K.fit(X_train, y_train)


TorchLSTMClassifier(epochs=500, hidden_layers=(128, 64, 32))

#### Inference on 20% dataset

In [40]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=clf_lstm_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    kcore_list = True,
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LSTM_results/tie_tol/50/expand_and_split/',
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LSTM_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 49.76%

Precision by Algorithm:
  UMKP: 15.00% (20 predictions)
  AltRB: 61.36% (44 predictions)
  kPEX: 62.16% (222 predictions)
  Pivot_Br: 23.68% (76 predictions)
  S_Br: 39.66% (58 predictions)

Recall by Algorithm:
  UMKP: 45.28% (24/53 cases)
  AltRB: 60.87% (42/69 cases)
  kPEX: 70.40% (157/223 cases)
  Pivot_Br: 62.58% (97/155 cases)
  S_Br: 51.37% (75/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 57210.80
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.233
  Avg Overhead/case ((pred - opt) / opt)    = 111.09%


Adding prediction probabilities...

Saving full predictions to './LSTM_results/tie_tol/50/expand_and_split/all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 211 wrong predictions with time metrics

Top 5 he

#### Re-evalute on entire dataset

In [41]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_lstm_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    kcore_list = True,
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LSTM_results/tie_tol/50/expand_and_split/',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LSTM_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.26%

Precision by Algorithm:
  UMKP: 32.00% (25 predictions)
  AltRB: 54.81% (104 predictions)
  kPEX: 60.10% (386 predictions)
  Pivot_Br: 21.92% (146 predictions)
  S_Br: 53.54% (99 predictions)

Recall by Algorithm:
  UMKP: 37.00% (37/100 cases)
  AltRB: 66.38% (77/116 cases)
  kPEX: 71.63% (260/363 cases)
  Pivot_Br: 60.62% (137/226 cases)
  S_Br: 50.39% (129/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 99143.90
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.482
  Avg Overhead/case ((pred - opt) / opt)    = 262.23%


Adding prediction probabilities...

Saving full predictions to './LSTM_results/tie_tol/50/expand_and_split/all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 378 wrong predictions with time metrics


## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


### **Balanced train-test split based on Y**

In [42]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 22)
Test set shape:  (156, 22)


In [43]:
import kplex_dev
import data_splits

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

X = X_train_.copy()

Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [45]:

# Instantiate and train
# Before calling fit():
clf_lstm_holdout_strat_y = TorchLSTMClassifier(hidden_layers=(128, 64, 32), lstm_hidden=32, epochs=500, temperature=1.0, lr= 0.01)

clf_lstm_holdout_strat_y.fit(X_train_, y_train_)



TorchLSTMClassifier(epochs=500, hidden_layers=(128, 64, 32))

#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_lstm_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    # drop_indices= [425],
    epsilon=1e-3,
    kcore_list = True,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LSTM_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LSTM_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 52.62%

Precision by Algorithm:
  UMKP: 50.00% (2 predictions)
  AltRB: 54.55% (77 predictions)
  kPEX: 61.48% (122 predictions)
  Pivot_Br: 46.21% (145 predictions)
  S_Br: 48.65% (74 predictions)

Recall by Algorithm:
  UMKP: 49.06% (26/53 cases)
  AltRB: 84.06% (58/69 cases)
  kPEX: 63.23% (141/223 cases)
  Pivot_Br: 71.61% (111/155 cases)
  S_Br: 52.05% (76/146 cases)

Time performance metrics:
Time Metrics:
  Total Predicted Time = 54373.90
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.172
  Avg Overhead/case ((pred - opt) / opt)    = 1723.42%


Adding prediction probabilities...

Saving full predictions to './LSTM_results/tie_tol/50/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 199 wrong predictions with time metrics

Top 5 heaviest wrong predictions:
           dataset  k_value 

#### Re-evalute on entire dataset

In [47]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_lstm_holdout_strat_y,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    kcore_list = True,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LSTM_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LSTM_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 49.87%

Precision by Algorithm:
  UMKP: 25.00% (8 predictions)
  AltRB: 49.26% (136 predictions)
  kPEX: 58.65% (237 predictions)
  Pivot_Br: 39.68% (247 predictions)
  S_Br: 55.30% (132 predictions)

Recall by Algorithm:
  UMKP: 36.00% (36/100 cases)
  AltRB: 73.28% (85/116 cases)
  kPEX: 63.36% (230/363 cases)
  Pivot_Br: 67.70% (153/226 cases)
  S_Br: 51.56% (132/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 99232.10
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.484
  Avg Overhead/case ((pred - opt) / opt)    = 289.29%


Adding prediction probabilities...

Saving full predictions to './LSTM_results/tie_tol/50/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y.csv'

Generating wrong-predictions with time metrics...

Saved 381 wrong predictions with time metrics

T

Additional time metrics:

- Total prediction time: sum of all runtimes for the algorithms our model actually chose, across every test instance
- Total Optimal Time:  sum of the runtime of the best (fastest) algorithm for each instance.
- Time Ratio (Predicted / Optimal)
- Average Overhead per Case: (predicted_time – optimal_time) / optimal_time + epsilon(0.001)

### **Balanced train-test split based on Y and K**

In [48]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (612, 22)
Test set shape:  (148, 22)


In [49]:
import kplex_dev
import data_splits

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance=0.50  # algorithms within tie_tolerance% of min treated as ties
    
)

# Label‐encode the string targets into integers 0…4
le = LabelEncoder()
y_train_ = le.fit_transform(y_train_)     # e.g. ['AltRB','UMKP',…] → [0,3,…]

print("Label ↔ Algorithm mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} ➔ {label}")

X = X_train_.copy()

Label ↔ Algorithm mapping:
  0 ➔ AltRB
  1 ➔ Pivot_Br
  2 ➔ S_Br
  3 ➔ UMKP
  4 ➔ kPEX


In [50]:
# Instantiate and train
# Before calling fit():
clf_lstm_holdout_strat_y_K = TorchLSTMClassifier(hidden_layers=(128, 64, 32), lstm_hidden=32, epochs=1000, temperature=1.0, lr= 0.01)

clf_lstm_holdout_strat_y_K.fit(X_train_, y_train_)


TorchLSTMClassifier(epochs=1000, hidden_layers=(128, 64, 32))

#### Inference on 20% dataset

In [ ]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=clf_lstm_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LSTM_results/tie_tol/50/split_and_expand',
    kcore_list = True,
    file_suffix= "inference_test_set_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LSTM_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 51.19%

Precision by Algorithm:
  UMKP: 50.00% (4 predictions)
  AltRB: 55.38% (65 predictions)
  kPEX: 64.29% (196 predictions)
  Pivot_Br: 27.27% (77 predictions)
  S_Br: 38.46% (78 predictions)

Recall by Algorithm:
  UMKP: 49.06% (26/53 cases)
  AltRB: 73.91% (51/69 cases)
  kPEX: 68.16% (152/223 cases)
  Pivot_Br: 60.65% (94/155 cases)
  S_Br: 54.11% (79/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 55153.20
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.189
  Avg Overhead/case ((pred - opt) / opt)    = 124.83%


Adding prediction probabilities...

Saving full predictions to './LSTM_results/tie_tol/50/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 205 wrong predictions with time metrics

Top 5 hea

#### Re-evalute on entire dataset

In [52]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=clf_lstm_holdout_strat_y_K,
    le=le,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./LSTM_results/tie_tol/50/split_and_expand',
    kcore_list = True,
    file_suffix= "inference_entire_data_without_holdout_strat_y_K" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./LSTM_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.92%

Precision by Algorithm:
  UMKP: 50.00% (14 predictions)
  AltRB: 52.07% (121 predictions)
  kPEX: 60.94% (361 predictions)
  Pivot_Br: 23.02% (126 predictions)
  S_Br: 49.28% (138 predictions)

Recall by Algorithm:
  UMKP: 39.00% (39/100 cases)
  AltRB: 72.41% (84/116 cases)
  kPEX: 69.42% (252/363 cases)
  Pivot_Br: 57.08% (129/226 cases)
  S_Br: 54.30% (139/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 96591.70
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.444
  Avg Overhead/case ((pred - opt) / opt)    = 257.20%


Adding prediction probabilities...

Saving full predictions to './LSTM_results/tie_tol/50/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y_K.csv'

Generating wrong-predictions with time metrics...

Saved 373 wrong predictions with time metrics